# Official task token comparison
The former GitHub-context plot is now an alias for the official SWE-Bench Lite + Terminal-Bench evaluation.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
PAPER = ROOT / 'paper' / 'img'
PAPER.mkdir(parents=True, exist_ok=True)
frames = []
for backend in ('deepseek_harness', 'codex', 'official'):
    path = RESULTS / backend / 'official_token_summary.csv'
    if path.exists(): frames.append(pd.read_csv(path))
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
if not df.empty:
    view = df.groupby(['suite', 'scale', 'mode'], as_index=False)['total_tokens_mean'].mean()
    fig, ax = plt.subplots(figsize=(6.8, 3.0), dpi=300)
    for (suite, mode), group in view.groupby(['suite', 'mode']):
        group = group.sort_values('scale')
        ax.plot(group['scale'], group['total_tokens_mean'], marker='o', linewidth=1.0, label=f'{suite}:{mode}')
    ax.set_xlabel('Official task scale')
    ax.set_ylabel('Total tokens (mean)')
    ax.legend(fontsize=6, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGDIR / 'FIG-Official-Token-Tasks.pdf', bbox_inches='tight')
    fig.savefig(PAPER / 'FIG-Official-Token-Tasks.pdf', bbox_inches='tight')
else:
    print('No official summary found; run bench_official_tasks.py first.')
# Historical github_token_tasks_raw.csv is provenance only; it is never loaded here.
